# Stage 3: Custom Biomedical Tokenizer

This notebook trains domain-specific BPE tokenizers on PubMedQA unlabeled text.

Outputs:
- `../tokenizers/biomedical_bpe_30k.json`
- `../tokenizers/biomedical_bpe_50k.json`
- `../results/stage3_tokenizer_training_summary.json`
- `../results/stage3_tokenizer_vocab_preview.csv`

In [1]:
import json
import os
from datetime import datetime

import pandas as pd
from datasets import load_dataset
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

In [2]:
RANDOM_SEED = 42
MAX_TRAIN_SAMPLES = 61249
VOCAB_SIZES = [30000, 50000, 100000]
MIN_FREQUENCY = 10

RESULTS_DIR = "../results"
TOKENIZERS_DIR = "../tokenizers"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TOKENIZERS_DIR, exist_ok=True)

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

# Use conservative normalization and whitespace-based pre-tokenization.
# This allows BPE to learn larger merges instead of forcing aggressive splits.
PRETOKENIZATION_PATTERN = "Whitespace"

In [3]:
def get_context_text(context_field):
    if isinstance(context_field, dict):
        contexts = context_field.get("contexts", [])
        if isinstance(contexts, list):
            return " ".join(str(part) for part in contexts)
    if isinstance(context_field, list):
        return " ".join(str(part) for part in context_field)
    return str(context_field)


def build_training_texts(max_samples=MAX_TRAIN_SAMPLES):
    ds = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled")["train"]
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=RANDOM_SEED).select(range(max_samples))

    df = pd.DataFrame(
        {
            "question": ds["question"],
            "context_text": [get_context_text(x) for x in ds["context"]],
        }
    )
    texts = (df["question"].fillna("") + " [SEP] " + df["context_text"].fillna("")).tolist()
    return texts


def train_bpe_tokenizer(texts, vocab_size):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=MIN_FREQUENCY,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer


def save_vocab_preview(tokenizer_name, tokenizer_obj, top_k=50):
    vocab = tokenizer_obj.get_vocab()
    sorted_items = sorted(vocab.items(), key=lambda x: x[1])[:top_k]
    return pd.DataFrame(
        {
            "tokenizer": tokenizer_name,
            "token": [token for token, _ in sorted_items],
            "token_id": [idx for _, idx in sorted_items],
        }
    )

In [4]:
training_texts = build_training_texts(MAX_TRAIN_SAMPLES)
print(f"Training corpus size: {len(training_texts)} samples")

Training corpus size: 61249 samples


In [5]:
trained_tokenizers = {}
summary_rows = []
preview_tables = []

for vocab_size in VOCAB_SIZES:
    name = f"biomedical_bpe_{int(vocab_size/1000)}k"
    output_path = os.path.join(TOKENIZERS_DIR, f"{name}.json")

    tokenizer_obj = train_bpe_tokenizer(training_texts, vocab_size=vocab_size)
    tokenizer_obj.save(output_path)
    trained_tokenizers[name] = tokenizer_obj

    actual_vocab_size = len(tokenizer_obj.get_vocab())
    summary_rows.append(
        {
            "tokenizer_name": name,
            "requested_vocab_size": vocab_size,
            "actual_vocab_size": actual_vocab_size,
            "min_frequency": MIN_FREQUENCY,
            "special_tokens_count": len(SPECIAL_TOKENS),
            "output_path": output_path,
        }
    )

    preview_tables.append(save_vocab_preview(name, tokenizer_obj, top_k=50))
    print(f"Saved {name} to {output_path} (vocab: {actual_vocab_size})")

summary_df = pd.DataFrame(summary_rows)
summary_df




Saved biomedical_bpe_30k to ../tokenizers/biomedical_bpe_30k.json (vocab: 30000)



Saved biomedical_bpe_50k to ../tokenizers/biomedical_bpe_50k.json (vocab: 39681)



Saved biomedical_bpe_100k to ../tokenizers/biomedical_bpe_100k.json (vocab: 39681)


,tokenizer_name,requested_vocab_size,actual_vocab_size,min_frequency,special_tokens_count,output_path
0,biomedical_bpe_30k,30000,30000,10,5,../tokenizers/biomedical_bpe_30k.json
1,biomedical_bpe_50k,50000,39681,10,5,../tokenizers/biomedical_bpe_50k.json
2,biomedical_bpe_100k,100000,39681,10,5,../tokenizers/biomedical_bpe_100k.json


In [6]:
summary_path = os.path.join(RESULTS_DIR, "stage3_tokenizer_training_summary.json")
preview_path = os.path.join(RESULTS_DIR, "stage3_tokenizer_vocab_preview.csv")

summary_payload = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "training_dataset": "qiaojin/PubMedQA::pqa_unlabeled",
    "training_samples": len(training_texts),
    "min_frequency": MIN_FREQUENCY,
    "special_tokens": SPECIAL_TOKENS,
    "pretokenization_pattern": PRETOKENIZATION_PATTERN,
    "tokenizers": summary_rows,
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

pd.concat(preview_tables, ignore_index=True).to_csv(preview_path, index=False)

print(f"Saved summary: {summary_path}")
print(f"Saved vocabulary preview: {preview_path}")

Saved summary: ../results/stage3_tokenizer_training_summary.json
Saved vocabulary preview: ../results/stage3_tokenizer_vocab_preview.csv


/tmp/ipykernel_28657/4028802308.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",


In [7]:
test_terms = [
    "gastroesophageal",
    "nephrolithiasis",
    "hypercholesterolemia",
    "electrocardiogram",
    "immunohistochemistry",
]

rows = []
for tokenizer_name, tokenizer_obj in trained_tokenizers.items():
    for term in test_terms:
        encoding = tokenizer_obj.encode(term)
        rows.append(
            {
                "tokenizer": tokenizer_name,
                "term": term,
                "pieces": encoding.tokens,
                "n_pieces": len(encoding.tokens),
            }
        )

term_splits_df = pd.DataFrame(rows)
term_splits_df

,tokenizer,term,pieces,n_pieces
0,biomedical_bpe_30k,gastroesophageal,[gastroesophageal],1
1,biomedical_bpe_30k,nephrolithiasis,[nephrolithiasis],1
2,biomedical_bpe_30k,hypercholesterolemia,[hypercholesterolemia],1
3,biomedical_bpe_30k,electrocardiogram,[electrocardiogram],1
4,biomedical_bpe_30k,immunohistochemistry,[immunohistochemistry],1
5,biomedical_bpe_50k,gastroesophageal,[gastroesophageal],1
6,biomedical_bpe_50k,nephrolithiasis,[nephrolithiasis],1
7,biomedical_bpe_50k,hypercholesterolemia,[hypercholesterolemia],1
8,biomedical_bpe_50k,electrocardiogram,[electrocardiogram],1
9,biomedical_bpe_50k,immunohistochemistry,[immunohistochemistry],1
